# 09 - RAG Q&A (Ollama + LangChain)

**Goal**: Ask natural-language questions and answer them using retrieved job postings.

**Retrieval backends**:
- FAISS (local index built in notebook 07)
- pgvector (database built in notebook 08)

**Note**: This notebook uses Ollama locally. Ensure the Ollama daemon is running and a model is pulled.

---

In [ ]:
print(f"Importing libraries and modules...")

FORCE_RECOMPUTE = False  # Placeholder for consistency with earlier notebooks

import numpy as np
import pandas as pd

from talentlens.config import (
    POSTINGS_NLP_PARQUET,
    EMBEDDINGS_NPY,
    DATABASE_URL,
)
from talentlens.rag import (
    FaissRetriever,
    PgvectorRetriever,
    answer_question,
)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 200)

print(f"[SUCCESS] Libraries imported successfully.")

## 1. Choose retrieval backend

Set `BACKEND` to `faiss` (local) or `pgvector` (database).

In [ ]:
BACKEND = 'faiss'  # 'faiss' or 'pgvector'
OLLAMA_MODEL = 'llama3.1'

if BACKEND == 'faiss':
    retriever = FaissRetriever()
    print('[SUCCESS] Using FAISS retriever')
elif BACKEND == 'pgvector':
    print(f"DATABASE_URL set: {bool(DATABASE_URL)}")
    retriever = PgvectorRetriever()
    print('[SUCCESS] Using pgvector retriever')
else:
    raise ValueError('BACKEND must be faiss or pgvector')

## 2. Load a query embedding

For a simple demo we’ll pick a job posting and use its embedding as the retrieval query.
In a full pipeline, you would embed the user question itself.

In [ ]:
if not POSTINGS_NLP_PARQUET.exists() or not EMBEDDINGS_NPY.exists():
    raise FileNotFoundError('Missing artifacts. Run notebooks 04 and 06 first.')

df = pd.read_parquet(POSTINGS_NLP_PARQUET)
embeddings = np.load(EMBEDDINGS_NPY)
print(f"Loaded df={len(df):,}, embeddings={embeddings.shape}")

query_idx = 0  # change this to test different postings
query_embedding = embeddings[query_idx]

display(df.iloc[[query_idx]][['job_id', 'title', 'location']])

## 3. Ask a question

Try questions aligned to your research themes:
- What skills are emphasized in these similar postings?
- Is remote work mentioned as a benefit or requirement?
- What seniority signals appear frequently?


In [ ]:
question = (
    "Summarize the common requirements and skills across these retrieved job postings. "
    "Call out any years-of-experience requirements, and cite job_ids."
)

result = answer_question(
    question=question,
    query_embedding=query_embedding,
    retriever=retriever,
    k=8,
    model=OLLAMA_MODEL,
)

print(result['answer'])

docs_df = pd.DataFrame(result['docs'])
display(docs_df[['job_id', 'score', 'title', 'location', 'content']])